# V42: Deterministic Hydrological Bucket Model 🪣💧
Ini adalah simulasi Fisika murni berbasis Konsep Reservoir (Model Ember). XGBoost dibuang sepenuhnya. Kita menggunakan Optuna hanya untuk 'Mengkalibrasi Parameter Alam' (seperti kapasitas serap tanah, kecepatan aliran sungai, dan *baseline* elevasi) agar simulasi matematika ini cocok dengan sejarah ketinggian air Bengawan Solo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# 1. Pemuatan Data
path = ''
train_df = pd.read_csv(path + 'train.csv', parse_dates=['datetime'])
test_df = pd.read_csv(path + 'test.csv')
env_df = pd.read_csv(path + 'data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
sample_sub = pd.read_csv(path + 'sample_submission.csv')

# Parsing Test Datetime
test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]

print(f"Data Loaded: Train {train_df.shape}, Test {test_parsed.shape}")

### Menyusun Timeline Kontinu (Wajib untuk Simulasi Fisika)
Simulasi fisika tidak bisa putus-putus. Air yang masuk hari ini akan berbekas untuk besok. Jadi kita gabungkan cuaca *Train* dan *Test* menjadi satu garis waktu tanpa putus.

In [ ]:
# Ekstraksi Fitur Cuaca Murni
env_df = env_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

# Membuat Timeline Kontinu (Train + Test)
stations = train_df['nama_pos'].unique()
timeline_dict = {}

for st in stations:
    # Ambil curah hujan
    env_st = env_df[env_df['nama_pos'] == st][['datetime', 'rainfall_mm']]
    
    # Ambil target TMA
    train_st = train_df[train_df['nama_pos'] == st][['datetime', 'tma_mdpl']]
    test_st = test_parsed[test_parsed['nama_pos'] == st][['datetime']]
    
    # Gabung cuaca dan target
    merged = pd.merge(env_st, train_st, on='datetime', how='left')
    
    # Tandai test set
    merged['is_test'] = merged['datetime'].isin(test_st['datetime'])
    
    # Urutkan kronologis
    merged = merged.sort_values('datetime').reset_index(drop=True)
    
    # Forward fill hujan jika bolong (asumsi kemarau/0 jika NaN)
    merged['rainfall_mm'] = merged['rainfall_mm'].fillna(0.0)
    
    timeline_dict[st] = merged
    
print("Timeline Kontinu Fisika Siap.")

### The Physics Engine: Simulasi Model Ember (Bucket Model)
Persamaan Diferensial Sederhana (Pseudo-HBV): 
1. **Runoff (Banjir):** Air hujan sebagian menyerap, sebagian meluber ke sungai. Semakin penuh ember tanah, semakin banyak air yang meluber.
2. **Routing (Aliran):** Air di sungai tidak diam, ia mengalir dengan koefisien pembuangan tertentu.
3. **Elevasi:** TMA (Tinggi Muka Air) adalah *Base Elevation* ditambah volume air di palung sungai.

In [ ]:
# Fungsi Matematika Deterministik Murni (Tidak ada Machine Learning)
def simulate_river_physics(rainfall_series, S_max, beta, leakage, K_r, base_elev, C):
    n_steps = len(rainfall_series)
    TMA_pred = np.zeros(n_steps)
    
    # Initial States (Kondisi Awal Lingkungan)
    soil_storage = S_max * 0.5  # Tanah dianggap setengah basah di awal waktu
    river_storage = 0.0         # Volume tambahan sungai nol di awal
    
    for i in range(n_steps):
        P = rainfall_series.iloc[i] # Hujan (mm)
        
        # 1. Soil Bucket & Runoff Generation (Non-linear HBV-style)
        # Rasio seberapa jenuh tanahnya
        saturation = soil_storage / S_max
        saturation = min(1.0, max(0.0, saturation))
        
        # Fraksi hujan yang langsung jadi Runoff (makin jenuh, makin besar)
        runoff_ratio = saturation ** beta
        runoff = P * runoff_ratio
        
        # Sisa hujan menyerap ke tanah
        infiltration = P - runoff
        soil_storage += infiltration
        
        # Tanah kehilangan air karena Evaporasi / Drainase bawah tanah (Leakage)
        soil_storage -= leakage
        soil_storage = min(S_max, max(0.0, soil_storage)) # Batasi 0 s.d S_max
        
        # 2. River Routing (Linear Reservoir)
        river_storage += runoff
        discharge = K_r * river_storage # Air mengalir/dibuang perlahan
        river_storage -= discharge
        
        # 3. Konversi ke Ketinggian (Elevasi)
        TMA_pred[i] = base_elev + (C * river_storage)
        
    return TMA_pred

### Kalibrasi Fisika Menggunakan Optuna
Optuna mencari 'Konstanta Alam' per stasiun agar grafik simulasi cocok dengan grafik aktual sejarah.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.ERROR)
N_TRIALS = 300 # Harus besar karena mencari parameter non-linear

calibrated_params = {}
calibrated_scores = {}
simulated_preds_full = {}

print(f"Memulai Kalibrasi Hukum Alam untuk 30 Stasiun (Total {30 * N_TRIALS} Evolusi)...\n")

for st in stations:
    df_st = timeline_dict[st]
    
    # Data Sejarah untuk kalibrasi (Hanya baris yang punya tma_mdpl aktual)
    train_mask = df_st['tma_mdpl'].notna() & (~df_st['is_test'])
    y_actual = df_st.loc[train_mask, 'tma_mdpl'].values
    
    # Hitung batas masuk akal untuk Base Elevation
    min_tma = y_actual.min() if len(y_actual) > 0 else 0.0
    
    def objective(trial):
        # Menebak 'Konstanta Alam'
        S_max = trial.suggest_float('S_max', 50.0, 1000.0)      # Kapasitas penyerapan tanah (mm)
        beta = trial.suggest_float('beta', 0.5, 4.0)            # Koefisien bentuk jenuh tanah
        leakage = trial.suggest_float('leakage', 0.0, 15.0)     # Kecepatan air hilang/rembes (mm/hari)
        K_r = trial.suggest_float('K_r', 0.1, 1.0)              # Kecepatan sungai mengalir membuang air
        base_elev = trial.suggest_float('base_elev', min_tma - 2.0, min_tma + 2.0) # Titik nadir kemarau
        C = trial.suggest_float('C', 0.001, 1.0, log=True)      # Rasio Volume ke Elevasi Meter
        
        # Jalankan Simulasi Alam SEMENJAK AWAL WAKTU
        preds_all = simulate_river_physics(df_st['rainfall_mm'], S_max, beta, leakage, K_r, base_elev, C)
        
        # Ekstrak tebakan HANYA di tanggal Train (untuk diukur error-nya)
        preds_train = preds_all[train_mask]
        
        # Hitung RMSE
        rmse = np.sqrt(mean_squared_error(y_actual, preds_train))
        return rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    best_p = study.best_params
    calibrated_params[st] = best_p
    calibrated_scores[st] = study.best_value
    
    # Simulasi Final dengan parameter terbaik untuk MASA LALU + MASA DEPAN sekaligus!
    final_sim_all = simulate_river_physics(
        df_st['rainfall_mm'], 
        best_p['S_max'], best_p['beta'], best_p['leakage'], 
        best_p['K_r'], best_p['base_elev'], best_p['C']
    )
    
    simulated_preds_full[st] = final_sim_all
    
    print(f"[{st}] - RMSE Fisika: {study.best_value:.4f} | S_max: {best_p['S_max']:.1f} | Base_Elev: {best_p['base_elev']:.2f}")

global_rmse = np.mean(list(calibrated_scores.values()))
print(f"\n🏆 GRAND AVERAGE PHYSICAL RMSE ACROSS 30 STATIONS: {global_rmse:.4f}")

### Evaluasi Visual: Alam Tak Pernah Berbohong
Grafik di bawah bukan hasil hafalan Machine Learning, melainkan hasil simulasi air mengalir.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

stations_list = sorted(list(stations))

def plot_physics(station_name):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(15, 6))
    
    df_st = timeline_dict[station_name]
    train_mask = df_st['tma_mdpl'].notna() & (~df_st['is_test'])
    
    # Ambil 300 data sejarah terakhir untuk divisualisasi
    plot_idx = df_st[train_mask].index[-300:]
    
    actual_tma = df_st.loc[plot_idx, 'tma_mdpl']
    sim_tma = simulated_preds_full[station_name][plot_idx]
    rain = df_st.loc[plot_idx, 'rainfall_mm']
    
    ax.plot(actual_tma.values, label='Actual TMA', color='blue', linewidth=2.5, alpha=0.7)
    ax.plot(sim_tma, label='Simulated Physics (Bucket)', color='red', linestyle='--', linewidth=2, alpha=0.9)
    
    # Tambahkan curah hujan terbalik di atas grafik
    ax2 = ax.twinx()
    ax2.bar(range(len(rain)), rain.values, width=1.0, color='gray', alpha=0.3, label='Rainfall')
    ax2.set_ylim(max(rain)*4, 0) # Invert axis
    
    ax.set_title(f"Simulasi Fisika Deterministik [{station_name}]")
    ax.legend(loc='lower left')
    ax2.legend(loc='lower right')
    plt.show()
    
    print("Parameter Alam Terkalibrasi:")
    for k, v in calibrated_params[station_name].items():
        print(f" - {k}: {v:.4f}")

widgets.interactive(plot_physics, station_name=widgets.Dropdown(options=stations_list, description='Stasiun:'))

### Submission Generator
Menarik hasil simulasi murni dari zona *Test Set* ke format Kaggle.

In [ ]:
test_preds = []

for st in stations:
    df_st = timeline_dict[st]
    test_mask = df_st['is_test']
    
    test_dates = df_st.loc[test_mask, 'datetime']
    sim_preds = simulated_preds_full[st][test_mask]
    
    temp_df = pd.DataFrame({
        'id': test_dates.astype(str) + ' - ' + st,
        'tma_mdpl': sim_preds
    })
    test_preds.append(temp_df)

sub_pred = pd.concat(test_preds, ignore_index=True)
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')

# Forward/backward fill in case of any nan alignment issues
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

output_file = path + 'submission_v42_physics_bucket.csv'
sub.to_csv(output_file, index=False)
print(f"✅ File '{output_file}' berhasil dicetak! Siap disubmit ke Kaggle!")